In [1]:
import os
import sys
from pathlib import Path
import logging
import time

from websocket_server import WebsocketServer
from multiprocessing import Process

import pickle
import json
import pandas as pd
import numpy as np

cur_dir = os.getcwd()
path = Path(cur_dir)
sys.path.insert(0, str(path.parent.absolute()))

from src.preprocess import preprocess_df
from src.real_time_model import NetworkModel
from src.time_windowed import get_window


In [2]:


df_raw = pd.read_csv('..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Tuesday-WorkingHours.pcap_ISCX.csv' , header=0, encoding='cp1252')
df = preprocess_df(df_raw, date_col=' Timestamp')

df_temp = df.iloc[:10000,:]

In [28]:
df_temp

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
133037,192.168.10.25-54.192.37.17-54988-443-6,54.192.37.17,443,192.168.10.25,54988,6,2017-04-07 01:00:00,54,1,1,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
394262,192.168.10.9-87.250.250.119-5303-443-6,87.250.250.119,443,192.168.10.9,5303,6,2017-04-07 01:00:00,101114,2,1,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
132995,192.168.10.25-52.84.145.136-54978-443-6,192.168.10.25,54978,52.84.145.136,443,6,2017-04-07 01:00:00,169399,22,30,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
132994,192.168.10.9-87.250.250.119-5260-443-6,192.168.10.9,5260,87.250.250.119,443,6,2017-04-07 01:00:00,148,2,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
138980,192.168.10.12-199.182.221.110-123-123-17,192.168.10.12,123,199.182.221.110,123,17,2017-04-07 01:00:00,71771,1,1,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139316,157.240.2.25-192.168.10.17-443-54917-6,157.240.2.25,443,192.168.10.17,54917,6,2017-04-07 01:20:00,44,3,0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
139317,192.168.10.17-31.13.80.36-51782-443-6,192.168.10.17,51782,31.13.80.36,443,6,2017-04-07 01:20:00,4217364,5,1,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
347783,192.168.10.25-52.84.145.135-55479-443-6,52.84.145.135,443,192.168.10.25,55479,6,2017-04-07 01:20:00,231,1,1,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
139266,192.168.10.25-52.84.145.135-55480-443-6,192.168.10.25,55480,52.84.145.135,443,6,2017-04-07 01:20:00,167455,33,61,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [18]:
with open(r'saves\victim_net.pickle', 'rb') as handle:
   entity_names = pickle.load(handle) 

len(entity_names)

13

In [22]:
with open(r'saves\connected_84.pickle', 'rb') as handle:
   entity_names = pickle.load(handle) 

len(entity_names)

84

In [4]:


server = WebsocketServer(host='127.0.0.1', port=15674, loglevel=logging.INFO)
print("Starting AMQP Server")
server.run_forever(threaded=True)

INFO:websocket_server.websocket_server:Listening on port 15674 for clients..
INFO:websocket_server.websocket_server:Starting WebsocketServer on thread Thread-5 (serve_forever).


Starting AMQP Server


In [27]:



n_entities = len(entity_names)
t_sync, t_graph = 20, 800 # sec 2,5 00
t_disp = 1 # Amount of time the graph is displayed on the rendering end
ff = 0.5
rf = 0.7
display_time = 1

nm = NetworkModel(entity_names=entity_names, mat_x_init= np.ones(n_entities), mat_p_init=np.eye(n_entities))
date_col=' Timestamp'
end_of_df = False
i = 0
current_datetime = df.iloc[0][date_col]
last_datetime = df.iloc[-1][date_col]

print("Waiting for Client")

# Await until connection
while len(server.clients) == 0:
    time.sleep(1)
    continue

print("Client Connected")
jsonstring = json.dumps({'funcEdges': nm.mat_f.tolist(), 'risk_mean': {name: x for name, x in zip(nm.entity_names, nm.mat_x)},
                             'risk_cov': nm.mat_p.tolist()})
server.send_message(server.clients[0], jsonstring)
time.sleep(display_time)

while end_of_df is False:
    temp_df, current_datetime = get_window(current_datetime, df, date_col=date_col, time_window=t_graph,
                                                       time_scale='sec')
    if current_datetime >= last_datetime:
        end_of_df = True
    #if temp_df.empty or len(temp_df.shape) < 2 or temp_df.shape[0] < MIN_SAMPLES:
    #    continue

    i += 1
    print("Graph #" + str(i))

    nm.update_new_tick(temp_df, conn_param='NPR',  sync_window_size=t_sync, time_scale='sec', keep_unit=True, forget_factor=ff, relief_factor=rf)
    mat_x, mat_p, mat_f, names = nm.mat_x, nm.mat_p, nm.mat_f, nm.entity_names

    scale = mat_x.max() / 2
    mat_x = mat_x / scale
    mat_p = mat_p / scale ** 2

    # Send the results
    jsonstring = json.dumps({'funcEdges': mat_f.tolist(), 'risk_mean': {name: x for name, x in zip(names, mat_x)},
                             'risk_cov': mat_p.tolist()})

    server.send_message(server.clients[0], jsonstring)
    time.sleep(display_time)
    

Waiting for Client
Client Connected
Graph #1
Conditioning number:  2.006454053665686e+16 
Determinant of F^T*F:  1.3280437706242325e-45
Graph #2
Conditioning number:  3.6323387565794744e+16 
Determinant of F^T*F:  -4.2795644202645584e-48
Graph #3
Conditioning number:  16533.5432593483 
Determinant of F^T*F:  3.735448797876366e-12
Graph #4
Conditioning number:  1.738600168315123e+16 
Determinant of F^T*F:  4.336057944296944e-51
Graph #5
Conditioning number:  2835084864941755.5 
Determinant of F^T*F:  -7.767839807614254e-30
Graph #6
Conditioning number:  1.965918936896891e+17 
Determinant of F^T*F:  -1.835111009686472e-99
Graph #7
Conditioning number:  3.105719382724539e+17 
Determinant of F^T*F:  -1.7170281971257905e-94
Graph #8
Conditioning number:  7.198708949298877e+17 
Determinant of F^T*F:  -4.907051983819888e-167
Graph #9
Conditioning number:  2.5378932544489654e+17 
Determinant of F^T*F:  1.7811383151314066e-77
Graph #10
Conditioning number:  5268062734964900.0 
Determinant of F^

C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\numpy\lib\function_base.py:520: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\bayer\PycharmProjects\NRE\src\network_connectivity.py:520: RuntimeWarning: Degrees of freedom <= 0 for slice
  cov = np.cov(samples.T)
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(

ValueError: Input must be 1- or 2-d.

In [15]:
server.clients



[]

In [8]:
server.disconnect_clients_gracefully()

INFO:websocket_server.websocket_server:Client asked to close connection.


In [14]:
server.shutdown_gracefully()

In [33]:
server.shutdown_abruptly()

ERROR:websocket_server.websocket_server:********************************************************************************
Exception in child thread <WebsocketServerThread(Thread-9 (serve_forever), started daemon 33304)>: [WinError 10038] An operation was attempted on something that is not a socket
********************************************************************************
Traceback (most recent call last):
  File "C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\websocket_server\thread.py", line 27, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 232, in serve_forever
    ready = selector.select(poll_interval)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\selectors.py", line 324, in select
    r, w, _ = self._select(self._readers, self._writers, [], timeout)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\selectors.py", line 315, in _selec

In [21]:
del server